
Лабораторная работа 3: Подготовить jupyter-ноутбук с описанием решения. Просьба подробно всё пояснять с комментариями и выводами.

Взять стандартную архитектуру ResNet (например, ResNet18 или ResNet34) Обучить ее на выбранном Вами датасете (например, можете взять датасет со спортивными трансляциями). Учить с нуля не нужно, достаточно потюнить полносвязный слой. Модифицировать encoder ResNet-а, добавив в него механизм Self-Attention Обучить модифицированную архитектуру с Self-Attention на этом же датасете. Сравнить две модели на тестовой выборке из датасета. Параметры тестирования и сиды необходимо закрепить для проведения корректного сравнения. Составить таблицу с результами. Добавить выводы.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split

from sklearn.metrics import accuracy_score, classification_report
import os

In [2]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
SEED = 42
seed_everything(SEED)

Device: cuda


In [3]:
# ============================================
# Загрузка и подготовка Oxford-IIIT Pet Dataset
# ============================================

from torchvision.datasets import OxfordIIITPet
from torch.utils.data import random_split

# Размер изображения для ResNet18
IMG_SIZE = 224

# Нормализация для ImageNet-предобученной ResNet
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# --------------------------------------------
# Аугментации для train
# --------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# --------------------------------------------
# Для validation/test без случайных преобразований
# --------------------------------------------

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# --------------------------------------------
# Загружаем полный датасет
# --------------------------------------------

full_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=train_transform
)

num_classes = len(full_dataset.classes)

print(f"Количество изображений: {len(full_dataset)}")
print(f"Количество классов: {num_classes}")

# --------------------------------------------
# Train / Validation / Test
# --------------------------------------------

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)

# Для val и test отключаем аугментации
val_dataset.dataset.transform = test_transform
test_dataset.dataset.transform = test_transform

print("\nРазмеры выборок:")
print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Посмотрим список классов
print("\nПервые 10 классов:")
print(full_dataset.classes[:10])

100%|██████████| 792M/792M [00:36<00:00, 21.5MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 10.8MB/s]


Количество изображений: 3680
Количество классов: 37

Размеры выборок:
Train: 2576
Validation: 552
Test: 552

Первые 10 классов:
['Abyssinian', 'American Bulldog', 'American Pit Bull Terrier', 'Basset Hound', 'Beagle', 'Bengal', 'Birman', 'Bombay', 'Boxer', 'British Shorthair']


In [4]:
# ============================================
# Train / Validation / Test split
# Создание DataLoader
# ============================================

from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader, Subset

# --------------------------------------------
# Создаем два экземпляра датасета:
# train -> с аугментациями
# eval -> без аугментаций
# --------------------------------------------

train_dataset_full = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=False,
    transform=train_transform
)

eval_dataset_full = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=False,
    transform=test_transform
)

num_classes = len(train_dataset_full.classes)

# --------------------------------------------
# Формируем одинаковые индексы для всех выборок
# --------------------------------------------

dataset_size = len(train_dataset_full)

indices = torch.randperm(
    dataset_size,
    generator=torch.Generator().manual_seed(SEED)
).tolist()

train_size = int(0.70 * dataset_size)
val_size = int(0.15 * dataset_size)

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

# --------------------------------------------
# Создаем выборки
# --------------------------------------------

train_dataset = Subset(
    train_dataset_full,
    train_indices
)

val_dataset = Subset(
    eval_dataset_full,
    val_indices
)

test_dataset = Subset(
    eval_dataset_full,
    test_indices
)

print("Размеры выборок:")
print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")
print(f"Classes: {num_classes}")

# --------------------------------------------
# DataLoader
# --------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataLoader успешно созданы")

# --------------------------------------------
# Проверка одного батча
# --------------------------------------------

images, labels = next(iter(train_loader))

print("\nФорма батча изображений:", images.shape)
print("Форма батча меток:", labels.shape)
print("Минимальная метка:", labels.min().item())
print("Максимальная метка:", labels.max().item())

Размеры выборок:
Train: 2576
Validation: 552
Test: 552
Classes: 37

DataLoader успешно созданы

Форма батча изображений: torch.Size([32, 3, 224, 224])
Форма батча меток: torch.Size([32])
Минимальная метка: 0
Максимальная метка: 34


In [9]:
# ============================================
# Загрузка предобученной ResNet18
# Замена классификационной головы
# ============================================

from torchvision.models import resnet18, ResNet18_Weights

# --------------------------------------------
# Загружаем предобученную на ImageNet ResNet18
# --------------------------------------------

weights = ResNet18_Weights.IMAGENET1K_V1

model = resnet18(weights=weights)

print("Предобученная ResNet18 успешно загружена")

# --------------------------------------------
# Замораживаем encoder
# Будем обучать только классификатор
# --------------------------------------------

for param in model.parameters():
    param.requires_grad = False

# --------------------------------------------
# Заменяем последний полносвязный слой
# под количество классов Oxford-IIIT Pet
# --------------------------------------------

in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, num_classes)
)

# Разрешаем обучение только новой головы
for param in model.fc.parameters():
    param.requires_grad = True

# Переносим модель на GPU/CPU
model = model.to(DEVICE)

# --------------------------------------------
# Проверка архитектуры
# --------------------------------------------

print(model.fc)

# --------------------------------------------
# Подсчет параметров
# --------------------------------------------

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"\nВсего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")
print(
    f"Доля обучаемых параметров: "
    f"{100 * trainable_params / total_params:.2f}%"
)

# --------------------------------------------
# Проверочный forward pass
# --------------------------------------------

images, labels = next(iter(train_loader))

images = images.to(DEVICE)

with torch.no_grad():
    outputs = model(images)

print("\nРазмер входного батча:", images.shape)
print("Размер выхода модели:", outputs.shape)
print("Количество классов:", num_classes)

Предобученная ResNet18 успешно загружена
Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=512, out_features=37, bias=True)
)

Всего параметров: 11,195,493
Обучаемых параметров: 18,981
Доля обучаемых параметров: 0.17%

Размер входного батча: torch.Size([32, 3, 224, 224])
Размер выхода модели: torch.Size([32, 37])
Количество классов: 37


In [10]:
# ============================================
# Обучение базовой ResNet18
# ============================================

from copy import deepcopy
from tqdm import tqdm
import torch.optim as optim

# --------------------------------------------
# Гиперпараметры
# --------------------------------------------

NUM_EPOCHS = 10
LEARNING_RATE = 1e-3

# --------------------------------------------
# Функция потерь и оптимизатор
# Обучаем только классификационную голову
# --------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)

# --------------------------------------------
# Функции обучения и валидации
# --------------------------------------------

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, leave=False):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


# --------------------------------------------
# Цикл обучения
# --------------------------------------------

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_model_weights = deepcopy(model.state_dict())

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = deepcopy(model.state_dict())

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

# --------------------------------------------
# Загружаем лучшую модель
# --------------------------------------------

model.load_state_dict(best_model_weights)

print("\nОбучение завершено.")
print(f"Лучшая Validation Accuracy: {best_val_acc:.4f}")

Epoch [1/10] | Train Loss: 2.8962 | Train Acc: 0.2500 | Val Loss: 1.6324 | Val Acc: 0.6576


Epoch [2/10] | Train Loss: 1.7794 | Train Acc: 0.5532 | Val Loss: 0.9685 | Val Acc: 0.7899


Epoch [3/10] | Train Loss: 1.3908 | Train Acc: 0.6526 | Val Loss: 0.7577 | Val Acc: 0.7971


Epoch [4/10] | Train Loss: 1.2335 | Train Acc: 0.6759 | Val Loss: 0.5849 | Val Acc: 0.8569


Epoch [5/10] | Train Loss: 1.1102 | Train Acc: 0.6988 | Val Loss: 0.5489 | Val Acc: 0.8605


Epoch [6/10] | Train Loss: 1.0381 | Train Acc: 0.7131 | Val Loss: 0.4985 | Val Acc: 0.8569


Epoch [7/10] | Train Loss: 0.9996 | Train Acc: 0.7151 | Val Loss: 0.4947 | Val Acc: 0.8478


Epoch [8/10] | Train Loss: 0.9537 | Train Acc: 0.7236 | Val Loss: 0.4303 | Val Acc: 0.8822


Epoch [9/10] | Train Loss: 0.9049 | Train Acc: 0.7446 | Val Loss: 0.4295 | Val Acc: 0.8696


Epoch [10/10] | Train Loss: 0.9045 | Train Acc: 0.7384 | Val Loss: 0.4215 | Val Acc: 0.8659

Обучение завершено.
Лучшая Validation Accuracy: 0.8822


In [12]:
# ============================================
# Оценка базовой ResNet18 на тестовой выборке
# ============================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Переводим модель в режим инференса
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(DEVICE)

        outputs = model(images)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# --------------------------------------------
# Расчет метрик
# --------------------------------------------

test_accuracy = accuracy_score(all_labels, all_preds)

test_precision = precision_score(
    all_labels,
    all_preds,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    all_labels,
    all_preds,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    all_labels,
    all_preds,
    average="weighted",
    zero_division=0
)

# Сохраняем результаты базовой модели
baseline_results = {
    "Model": "ResNet18",
    "Accuracy": test_accuracy,
    "Precision": test_precision,
    "Recall": test_recall,
    "F1-score": test_f1
}

# --------------------------------------------
# Вывод результатов
# --------------------------------------------

print("=" * 50)
print("Результаты базовой ResNet18")
print("=" * 50)

print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1-score : {test_f1:.4f}")

# --------------------------------------------
# Подробный отчет по классам
# --------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_preds,
        zero_division=0
    )
)

# --------------------------------------------
# Таблица результатов
# --------------------------------------------

baseline_results_df = pd.DataFrame([baseline_results])

print("\nИтоговая таблица:")
baseline_results_df

100%|██████████| 18/18 [00:08<00:00,  2.14it/s]

Результаты базовой ResNet18
Accuracy : 0.8641
Precision: 0.8746
Recall   : 0.8641
F1-score : 0.8634

Classification Report:

              precision    recall  f1-score   support

           0       0.69      0.85      0.76        13
           1       0.75      0.94      0.83        16
           2       0.69      0.64      0.67        14
           3       0.73      0.92      0.81        12
           4       0.78      0.58      0.67        12
           5       0.79      0.79      0.79        14
           6       0.67      0.60      0.63        10
           7       0.91      1.00      0.95        10
           8       0.86      0.80      0.83        15
           9       0.75      0.86      0.80        14
          10       0.69      0.75      0.72        12
          11       0.84      0.94      0.89        17
          12       1.00      0.75      0.86        12
          13       0.85      1.00      0.92        11
          14       0.82      0.69      0.75        13
          

,Model,Accuracy,Precision,Recall,F1-score
0,ResNet18,0.86413,0.874555,0.86413,0.863438


In [13]:
# ============================================
# Self-Attention блок
# ============================================

import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    """
    Spatial Self-Attention для CNN.

    Вход:
        (B, C, H, W)

    Выход:
        (B, C, H, W)

    Размерность признаков сохраняется.
    """

    def __init__(self, in_channels):

        super().__init__()

        # Query
        self.query_conv = nn.Conv2d(
            in_channels,
            in_channels // 8,
            kernel_size=1
        )

        # Key
        self.key_conv = nn.Conv2d(
            in_channels,
            in_channels // 8,
            kernel_size=1
        )

        # Value
        self.value_conv = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=1
        )

        self.softmax = nn.Softmax(dim=-1)

        # Обучаемый коэффициент
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):

        batch_size, channels, height, width = x.size()

        # ----------------------------------
        # Query
        # (B,C,H,W) -> (B,HW,C')
        # ----------------------------------

        query = self.query_conv(x)

        query = query.view(
            batch_size,
            -1,
            height * width
        ).permute(0, 2, 1)

        # ----------------------------------
        # Key
        # (B,C,H,W) -> (B,C',HW)
        # ----------------------------------

        key = self.key_conv(x)

        key = key.view(
            batch_size,
            -1,
            height * width
        )

        # ----------------------------------
        # Attention map
        # (B,HW,HW)
        # ----------------------------------

        attention = torch.bmm(query, key)

        attention = self.softmax(attention)

        # ----------------------------------
        # Value
        # (B,C,HW)
        # ----------------------------------

        value = self.value_conv(x)

        value = value.view(
            batch_size,
            channels,
            height * width
        )

        # ----------------------------------
        # Взвешивание признаков
        # ----------------------------------

        out = torch.bmm(
            value,
            attention.permute(0, 2, 1)
        )

        out = out.view(
            batch_size,
            channels,
            height,
            width
        )

        # Residual connection
        out = self.gamma * out + x

        return out


# ============================================
# Проверка работоспособности блока
# ============================================

attention_block = SelfAttention(512).to(DEVICE)

dummy_input = torch.randn(
    2,
    512,
    7,
    7
).to(DEVICE)

dummy_output = attention_block(dummy_input)

print("Input shape :", dummy_input.shape)
print("Output shape:", dummy_output.shape)

assert dummy_input.shape == dummy_output.shape

print("\nSelf-Attention блок успешно создан.")

Input shape : torch.Size([2, 512, 7, 7])
Output shape: torch.Size([2, 512, 7, 7])

Self-Attention блок успешно создан.


In [14]:
# ============================================
# ResNet18 + Self-Attention
# Модификация encoder
# ============================================

from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

class ResNet18WithAttention(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        # ----------------------------------
        # Предобученная ResNet18
        # ----------------------------------

        backbone = resnet18(
            weights=ResNet18_Weights.IMAGENET1K_V1
        )

        # ----------------------------------
        # Encoder ResNet18
        # ----------------------------------

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # ----------------------------------
        # Self-Attention после layer4
        # ----------------------------------

        self.attention = SelfAttention(
            in_channels=512
        )

        # ----------------------------------
        # Head
        # ----------------------------------

        self.avgpool = backbone.avgpool

        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Encoder
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Self-Attention
        x = self.attention(x)

        # Classification head
        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x


# ============================================
# Создаем модель
# ============================================

attention_model = ResNet18WithAttention(
    num_classes=num_classes
)

# ============================================
# Замораживаем encoder
# (как и в базовой модели)
# ============================================

for param in attention_model.parameters():
    param.requires_grad = False

# Обучаем Self-Attention
for param in attention_model.attention.parameters():
    param.requires_grad = True

# Обучаем классификатор
for param in attention_model.fc.parameters():
    param.requires_grad = True

attention_model = attention_model.to(DEVICE)

# ============================================
# Проверка архитектуры
# ============================================

print(attention_model)

# ============================================
# Проверка числа параметров
# ============================================

total_params = sum(
    p.numel()
    for p in attention_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in attention_model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 50)
print("ResNet18 + Self-Attention")
print("=" * 50)

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")

# ============================================
# Проверочный forward pass
# ============================================

images, labels = next(iter(train_loader))

images = images.to(DEVICE)

with torch.no_grad():
    outputs = attention_model(images)

print("\nInput shape :", images.shape)
print("Output shape:", outputs.shape)

assert outputs.shape == (
    images.shape[0],
    num_classes
)

print("\nМодель успешно создана.")

ResNet18WithAttention(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [15]:
# ============================================
# Обучение ResNet18 + Self-Attention
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim

from copy import deepcopy
from tqdm import tqdm

# --------------------------------------------
# Гиперпараметры
# ДОЛЖНЫ совпадать с базовой моделью
# --------------------------------------------

NUM_EPOCHS = 8
LEARNING_RATE = 1e-3

# --------------------------------------------
# Функция потерь
# --------------------------------------------

criterion = nn.CrossEntropyLoss()

# --------------------------------------------
# Оптимизатор
# Обучаем только attention и fc
# --------------------------------------------

optimizer_attention = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        attention_model.parameters()
    ),
    lr=LEARNING_RATE
)

# --------------------------------------------
# История обучения
# --------------------------------------------

attention_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_attention_acc = 0.0
best_attention_weights = deepcopy(
    attention_model.state_dict()
)

# ============================================
# Цикл обучения
# ============================================

for epoch in range(NUM_EPOCHS):

    # ----------------------------------------
    # TRAIN
    # ----------------------------------------

    attention_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
        leave=False
    ):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_attention.zero_grad()

        outputs = attention_model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_attention.step()

        train_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # ----------------------------------------
    # VALIDATION
    # ----------------------------------------

    attention_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = attention_model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    # ----------------------------------------
    # Сохраняем историю
    # ----------------------------------------

    attention_history["train_loss"].append(
        epoch_train_loss
    )

    attention_history["train_acc"].append(
        epoch_train_acc
    )

    attention_history["val_loss"].append(
        epoch_val_loss
    )

    attention_history["val_acc"].append(
        epoch_val_acc
    )

    # ----------------------------------------
    # Сохраняем лучшую модель
    # ----------------------------------------

    if epoch_val_acc > best_attention_acc:

        best_attention_acc = epoch_val_acc

        best_attention_weights = deepcopy(
            attention_model.state_dict()
        )

    # ----------------------------------------
    # Логирование
    # ----------------------------------------

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.4f}"
    )

# ============================================
# Загружаем лучшую модель
# ============================================

attention_model.load_state_dict(
    best_attention_weights
)

print("\nОбучение завершено.")
print(
    f"Лучшая Validation Accuracy: "
    f"{best_attention_acc:.4f}"
)

Epoch [1/8] | Train Loss: 2.5549 | Train Acc: 0.3366 | Val Loss: 0.8465 | Val Acc: 0.7572


Epoch [2/8] | Train Loss: 1.0968 | Train Acc: 0.6894 | Val Loss: 0.5340 | Val Acc: 0.8188


Epoch [3/8] | Train Loss: 0.9645 | Train Acc: 0.7131 | Val Loss: 0.4194 | Val Acc: 0.8714


Epoch [4/8] | Train Loss: 0.8726 | Train Acc: 0.7356 | Val Loss: 0.4299 | Val Acc: 0.8496


Epoch [5/8] | Train Loss: 0.8553 | Train Acc: 0.7399 | Val Loss: 0.4279 | Val Acc: 0.8587


Epoch [6/8] | Train Loss: 0.7672 | Train Acc: 0.7636 | Val Loss: 0.4090 | Val Acc: 0.8641


Epoch [7/8] | Train Loss: 0.7581 | Train Acc: 0.7609 | Val Loss: 0.4067 | Val Acc: 0.8605


Epoch [8/8] | Train Loss: 0.7328 | Train Acc: 0.7706 | Val Loss: 0.4410 | Val Acc: 0.8460

Обучение завершено.
Лучшая Validation Accuracy: 0.8714


In [16]:
# ============================================
# Оценка ResNet18 + Self-Attention
# на тестовой выборке
# ============================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

attention_model.eval()

all_preds_att = []
all_labels_att = []

with torch.no_grad():

    for images, labels in tqdm(
        test_loader,
        desc="Testing Attention Model"
    ):

        images = images.to(DEVICE)

        outputs = attention_model(images)

        preds = torch.argmax(outputs, dim=1)

        all_preds_att.extend(
            preds.cpu().numpy()
        )

        all_labels_att.extend(
            labels.numpy()
        )

# ============================================
# Метрики
# ============================================

attention_accuracy = accuracy_score(
    all_labels_att,
    all_preds_att
)

attention_precision = precision_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

attention_recall = recall_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

attention_f1 = f1_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

# ============================================
# Сохраняем результаты
# ============================================

attention_results = {
    "Model": "ResNet18 + Self-Attention",
    "Accuracy": attention_accuracy,
    "Precision": attention_precision,
    "Recall": attention_recall,
    "F1-score": attention_f1
}

# ============================================
# Вывод результатов
# ============================================

print("=" * 60)
print("Результаты ResNet18 + Self-Attention")
print("=" * 60)

print(f"Accuracy : {attention_accuracy:.4f}")
print(f"Precision: {attention_precision:.4f}")
print(f"Recall   : {attention_recall:.4f}")
print(f"F1-score : {attention_f1:.4f}")

# ============================================
# Отчет по классам
# ============================================

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels_att,
        all_preds_att,
        zero_division=0
    )
)

# ============================================
# Таблица результатов
# ============================================

attention_results_df = pd.DataFrame(
    [attention_results]
)

print("\nИтоговая таблица:")

display(attention_results_df)

Testing Attention Model: 100%|██████████| 18/18 [00:04<00:00,  3.97it/s]


Результаты ResNet18 + Self-Attention
Accuracy : 0.8460
Precision: 0.8587
Recall   : 0.8460
F1-score : 0.8430

Classification Report:

              precision    recall  f1-score   support

           0       0.71      0.77      0.74        13
           1       0.72      0.81      0.76        16
           2       0.88      0.50      0.64        14
           3       0.89      0.67      0.76        12
           4       0.59      0.83      0.69        12
           5       0.60      0.86      0.71        14
           6       0.60      0.30      0.40        10
           7       0.71      1.00      0.83        10
           8       0.83      0.67      0.74        15
           9       1.00      0.64      0.78        14
          10       0.82      0.75      0.78        12
          11       0.93      0.76      0.84        17
          12       0.80      0.67      0.73        12
          13       0.82      0.82      0.82        11
          14       0.73      0.85      0.79        13
 

,Model,Accuracy,Precision,Recall,F1-score
0,ResNet18 + Self-Attention,0.846014,0.858698,0.846014,0.843042


In [17]:
print(attention_model.attention.gamma)

Parameter containing:
tensor([0.1267], device='cuda:0', requires_grad=True)


In [18]:
# ============================================
# Улучшенная ResNet18 + Self-Attention
#
# Изменения относительно предыдущей версии:
# 1. Attention переносится после layer3
# 2. Размораживается layer4
# 3. Разные learning rate
# 4. Больше эпох обучения
#
# Рекомендуется использовать вместо
# предыдущей attention_model
# ============================================

from torchvision.models import resnet18, ResNet18_Weights
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================
# Модель
# ============================================

class ResNet18WithAttention(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        backbone = resnet18(
            weights=ResNet18_Weights.IMAGENET1K_V1
        )

        # Stem
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        # Encoder
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # ----------------------------------
        # Attention после layer3
        # размер карты примерно 14x14
        # ----------------------------------

        self.attention = SelfAttention(
            in_channels=256
        )

        self.avgpool = backbone.avgpool

        self.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)

        x = self.layer3(x)

        # Self-Attention
        x = self.attention(x)

        x = self.layer4(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x


# ============================================
# Создание модели
# ============================================

attention_model = ResNet18WithAttention(
    num_classes=num_classes
)

# ============================================
# Замораживаем всё
# ============================================

for param in attention_model.parameters():
    param.requires_grad = False

# ============================================
# Размораживаем layer4
# ============================================

for param in attention_model.layer4.parameters():
    param.requires_grad = True

# ============================================
# Размораживаем Attention
# ============================================

for param in attention_model.attention.parameters():
    param.requires_grad = True

# ============================================
# Размораживаем классификатор
# ============================================

for param in attention_model.fc.parameters():
    param.requires_grad = True

attention_model = attention_model.to(DEVICE)

# ============================================
# Проверка
# ============================================

trainable_params = sum(
    p.numel()
    for p in attention_model.parameters()
    if p.requires_grad
)

print(f"Trainable params: {trainable_params:,}")

# ============================================
# Loss
# ============================================

criterion = nn.CrossEntropyLoss()

# ============================================
# Optimizer
# разные LR для разных частей модели
# ============================================

optimizer_attention = optim.Adam(

    [
        {
            "params": attention_model.layer4.parameters(),
            "lr": 1e-4
        },

        {
            "params": attention_model.attention.parameters(),
            "lr": 1e-3
        },

        {
            "params": attention_model.fc.parameters(),
            "lr": 1e-3
        }
    ]
)

# ============================================
# Scheduler
# ============================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_attention,
    mode="max",
    factor=0.5,
    patience=2
)

# ============================================
# Параметры обучения
# ============================================

NUM_EPOCHS = 15

print("Модель готова к обучению.")

Trainable params: 8,494,950
Модель готова к обучению.


In [19]:
# ============================================
# Обучение ResNet18 + Self-Attention
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim

from copy import deepcopy
from tqdm import tqdm

# --------------------------------------------
# Гиперпараметры
# ДОЛЖНЫ совпадать с базовой моделью
# --------------------------------------------

NUM_EPOCHS = 15
LEARNING_RATE = 1e-3

# --------------------------------------------
# Функция потерь
# --------------------------------------------

criterion = nn.CrossEntropyLoss()

# --------------------------------------------
# Оптимизатор
# Обучаем только attention и fc
# --------------------------------------------

optimizer_attention = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        attention_model.parameters()
    ),
    lr=LEARNING_RATE
)

# --------------------------------------------
# История обучения
# --------------------------------------------

attention_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_attention_acc = 0.0
best_attention_weights = deepcopy(
    attention_model.state_dict()
)

# ============================================
# Цикл обучения
# ============================================

for epoch in range(NUM_EPOCHS):

    # ----------------------------------------
    # TRAIN
    # ----------------------------------------

    attention_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
        leave=False
    ):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_attention.zero_grad()

        outputs = attention_model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_attention.step()

        train_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # ----------------------------------------
    # VALIDATION
    # ----------------------------------------

    attention_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = attention_model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    # ----------------------------------------
    # Сохраняем историю
    # ----------------------------------------

    attention_history["train_loss"].append(
        epoch_train_loss
    )

    attention_history["train_acc"].append(
        epoch_train_acc
    )

    attention_history["val_loss"].append(
        epoch_val_loss
    )

    attention_history["val_acc"].append(
        epoch_val_acc
    )

    # ----------------------------------------
    # Сохраняем лучшую модель
    # ----------------------------------------

    if epoch_val_acc > best_attention_acc:

        best_attention_acc = epoch_val_acc

        best_attention_weights = deepcopy(
            attention_model.state_dict()
        )

    # ----------------------------------------
    # Логирование
    # ----------------------------------------

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.4f}"
    )

    scheduler.step(epoch_val_acc)

# ============================================
# Загружаем лучшую модель
# ============================================

attention_model.load_state_dict(
    best_attention_weights
)

print("\nОбучение завершено.")
print(
    f"Лучшая Validation Accuracy: "
    f"{best_attention_acc:.4f}"
)

Epoch [1/15] | Train Loss: 1.8007 | Train Acc: 0.4895 | Val Loss: 1.0051 | Val Acc: 0.7156


Epoch [2/15] | Train Loss: 1.2336 | Train Acc: 0.6425 | Val Loss: 0.6601 | Val Acc: 0.7736


Epoch [3/15] | Train Loss: 1.0171 | Train Acc: 0.6964 | Val Loss: 0.6666 | Val Acc: 0.7899


Epoch [4/15] | Train Loss: 0.9055 | Train Acc: 0.7252 | Val Loss: 0.5930 | Val Acc: 0.8043


Epoch [5/15] | Train Loss: 0.8378 | Train Acc: 0.7422 | Val Loss: 0.5681 | Val Acc: 0.8315


...

Epoch [11/15] | Train Loss: 0.6199 | Train Acc: 0.8129 | Val Loss: 0.5794 | Val Acc: 0.8370


Epoch [12/15] | Train Loss: 0.5749 | Train Acc: 0.8253 | Val Loss: 0.5750 | Val Acc: 0.8152


Epoch [13/15] | Train Loss: 0.5638 | Train Acc: 0.8269 | Val Loss: 0.5969 | Val Acc: 0.8080


Epoch [14/15] | Train Loss: 0.5402 | Train Acc: 0.8389 | Val Loss: 0.4835 | Val Acc: 0.8533


Epoch [15/15] | Train Loss: 0.4948 | Train Acc: 0.8482 | Val Loss: 0.6652 | Val Acc: 0.8098

Обучение завершено.
Лучшая Validation Accuracy: 0.8533


In [20]:
# ============================================
# Оценка ResNet18 + Self-Attention
# на тестовой выборке
# ============================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

attention_model.eval()

all_preds_att = []
all_labels_att = []

with torch.no_grad():

    for images, labels in tqdm(
        test_loader,
        desc="Testing Attention Model"
    ):

        images = images.to(DEVICE)

        outputs = attention_model(images)

        preds = torch.argmax(outputs, dim=1)

        all_preds_att.extend(
            preds.cpu().numpy()
        )

        all_labels_att.extend(
            labels.numpy()
        )

# ============================================
# Метрики
# ============================================

attention_accuracy = accuracy_score(
    all_labels_att,
    all_preds_att
)

attention_precision = precision_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

attention_recall = recall_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

attention_f1 = f1_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

# ============================================
# Сохраняем результаты
# ============================================

attention_results = {
    "Model": "ResNet18 + Self-Attention",
    "Accuracy": attention_accuracy,
    "Precision": attention_precision,
    "Recall": attention_recall,
    "F1-score": attention_f1
}

# ============================================
# Вывод результатов
# ============================================

print("=" * 60)
print("Результаты ResNet18 + Self-Attention")
print("=" * 60)

print(f"Accuracy : {attention_accuracy:.4f}")
print(f"Precision: {attention_precision:.4f}")
print(f"Recall   : {attention_recall:.4f}")
print(f"F1-score : {attention_f1:.4f}")

# ============================================
# Отчет по классам
# ============================================

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels_att,
        all_preds_att,
        zero_division=0
    )
)

# ============================================
# Таблица результатов
# ============================================

attention_results_df = pd.DataFrame(
    [attention_results]
)

print("\nИтоговая таблица:")

display(attention_results_df)

Testing Attention Model: 100%|██████████| 18/18 [00:04<00:00,  4.01it/s]


Результаты ResNet18 + Self-Attention
Accuracy : 0.8496
Precision: 0.8703
Recall   : 0.8496
F1-score : 0.8494

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.92      0.89        13
           1       0.69      0.56      0.62        16
           2       0.88      0.50      0.64        14
           3       0.83      0.83      0.83        12
           4       0.67      0.67      0.67        12
           5       0.68      0.93      0.79        14
           6       0.75      0.90      0.82        10
           7       1.00      0.90      0.95        10
           8       0.73      0.73      0.73        15
           9       0.58      1.00      0.74        14
          10       0.86      0.50      0.63        12
          11       1.00      0.82      0.90        17
          12       1.00      0.75      0.86        12
          13       0.69      1.00      0.81        11
          14       0.85      0.85      0.85        13
 

,Model,Accuracy,Precision,Recall,F1-score
0,ResNet18 + Self-Attention,0.849638,0.870332,0.849638,0.849366


In [21]:
# ============================================
# ResNet18 + Self-Attention (v2)
#
# Attention после layer2
# Fine-tuning layer3 + layer4
# ============================================

from torchvision.models import resnet18, ResNet18_Weights
import torch
import torch.nn as nn
import torch.optim as optim

class ResNet18WithAttentionV2(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        backbone = resnet18(
            weights=ResNet18_Weights.IMAGENET1K_V1
        )

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # ----------------------------------
        # Attention после layer2
        # Размер карты примерно 28x28
        # Каналов = 128
        # ----------------------------------

        self.attention = SelfAttention(
            in_channels=128
        )

        self.avgpool = backbone.avgpool

        self.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)

        x = self.layer2(x)

        # Self-Attention
        x = self.attention(x)

        x = self.layer3(x)

        x = self.layer4(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x


# ============================================
# Создаем модель
# ============================================

attention_model = ResNet18WithAttentionV2(
    num_classes=num_classes
)

# ============================================
# Замораживаем всё
# ============================================

for param in attention_model.parameters():
    param.requires_grad = False

# ============================================
# Размораживаем layer3
# ============================================

for param in attention_model.layer3.parameters():
    param.requires_grad = True

# ============================================
# Размораживаем layer4
# ============================================

for param in attention_model.layer4.parameters():
    param.requires_grad = True

# ============================================
# Размораживаем Attention
# ============================================

for param in attention_model.attention.parameters():
    param.requires_grad = True

# ============================================
# Размораживаем FC
# ============================================

for param in attention_model.fc.parameters():
    param.requires_grad = True

attention_model = attention_model.to(DEVICE)

# ============================================
# Проверка количества параметров
# ============================================

trainable_params = sum(
    p.numel()
    for p in attention_model.parameters()
    if p.requires_grad
)

print(f"Trainable params: {trainable_params:,}")

# ============================================
# Loss
# ============================================

criterion = nn.CrossEntropyLoss()

# ============================================
# Optimizer
# ============================================

optimizer_attention = optim.Adam(

    [
        {
            "params": attention_model.layer3.parameters(),
            "lr": 1e-5
        },

        {
            "params": attention_model.layer4.parameters(),
            "lr": 1e-5
        },

        {
            "params": attention_model.attention.parameters(),
            "lr": 1e-3
        },

        {
            "params": attention_model.fc.parameters(),
            "lr": 1e-3
        }
    ],

    weight_decay=1e-4
)

# ============================================
# Scheduler
# ============================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_attention,
    mode="max",
    factor=0.5,
    patience=2
)

# ============================================
# Параметры обучения
# ============================================

NUM_EPOCHS = 20

print("Модель готова к обучению.")

Trainable params: 10,533,062
Модель готова к обучению.


In [22]:
# ============================================
# Обучение ResNet18 + Self-Attention (v2)
# ============================================

import torch
from copy import deepcopy
from tqdm import tqdm

# --------------------------------------------
# История обучения
# --------------------------------------------

attention_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_attention_acc = 0.0
best_attention_weights = deepcopy(
    attention_model.state_dict()
)

# ============================================
# Цикл обучения
# ============================================

for epoch in range(NUM_EPOCHS):

    # ========================================
    # TRAIN
    # ========================================

    attention_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}",
        leave=False
    )

    for images, labels in train_bar:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_attention.zero_grad()

        outputs = attention_model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_attention.step()

        train_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        train_correct += (
            preds == labels
        ).sum().item()

        train_total += labels.size(0)

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # ========================================
    # VALIDATION
    # ========================================

    attention_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = attention_model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            preds = outputs.argmax(dim=1)

            val_correct += (
                preds == labels
            ).sum().item()

            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    # ========================================
    # Scheduler
    # ========================================

    scheduler.step(epoch_val_acc)

    # ========================================
    # Сохраняем историю
    # ========================================

    attention_history["train_loss"].append(
        epoch_train_loss
    )

    attention_history["train_acc"].append(
        epoch_train_acc
    )

    attention_history["val_loss"].append(
        epoch_val_loss
    )

    attention_history["val_acc"].append(
        epoch_val_acc
    )

    # ========================================
    # Лучшая модель
    # ========================================

    if epoch_val_acc > best_attention_acc:

        best_attention_acc = epoch_val_acc

        best_attention_weights = deepcopy(
            attention_model.state_dict()
        )

    # ========================================
    # Логирование
    # ========================================

    current_lr = optimizer_attention.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | "
        f"LR={current_lr:.6f} | "
        f"Train Loss={epoch_train_loss:.4f} | "
        f"Train Acc={epoch_train_acc:.4f} | "
        f"Val Loss={epoch_val_loss:.4f} | "
        f"Val Acc={epoch_val_acc:.4f}"
    )

# ============================================
# Загружаем лучшую модель
# ============================================

attention_model.load_state_dict(
    best_attention_weights
)

print("\n" + "=" * 60)
print("Обучение завершено")
print("=" * 60)
print(
    f"Best Validation Accuracy: "
    f"{best_attention_acc:.4f}"
)

# ============================================
# Сохраняем веса
# ============================================

torch.save(
    attention_model.state_dict(),
    "best_attention_model.pth"
)

print(
    "Веса сохранены в "
    "'best_attention_model.pth'"
)

Epoch [01/20] | LR=0.000010 | Train Loss=2.8631 | Train Acc=0.2504 | Val Loss=1.3003 | Val Acc=0.7681


Epoch [02/20] | LR=0.000010 | Train Loss=1.5953 | Train Acc=0.5827 | Val Loss=0.6786 | Val Acc=0.8551


Epoch [03/20] | LR=0.000010 | Train Loss=1.1383 | Train Acc=0.6957 | Val Loss=0.4950 | Val Acc=0.8841


Epoch [04/20] | LR=0.000010 | Train Loss=0.9472 | Train Acc=0.7321 | Val Loss=0.4288 | Val Acc=0.8822


Epoch [05/20] | LR=0.000010 | Train Loss=0.8641 | Train Acc=0.7488 | Val Loss=0.3586 | Val Acc=0.8931


...

Epoch [16/20] | LR=0.000005 | Train Loss=0.5433 | Train Acc=0.8354 | Val Loss=0.2365 | Val Acc=0.9275


Epoch [17/20] | LR=0.000005 | Train Loss=0.5134 | Train Acc=0.8405 | Val Loss=0.2399 | Val Acc=0.9112


Epoch [18/20] | LR=0.000005 | Train Loss=0.5054 | Train Acc=0.8478 | Val Loss=0.2485 | Val Acc=0.9185


Epoch [19/20] | LR=0.000003 | Train Loss=0.5150 | Train Acc=0.8451 | Val Loss=0.2345 | Val Acc=0.9221


Epoch [20/20] | LR=0.000003 | Train Loss=0.4908 | Train Acc=0.8474 | Val Loss=0.2360 | Val Acc=0.9185

Обучение завершено
Best Validation Accuracy: 0.9275
Веса сохранены в 'best_attention_model.pth'


In [23]:
# ============================================
# Оценка ResNet18 + Self-Attention
# на тестовой выборке
# ============================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

attention_model.eval()

all_preds_att = []
all_labels_att = []

with torch.no_grad():

    for images, labels in tqdm(
        test_loader,
        desc="Testing Attention Model"
    ):

        images = images.to(DEVICE)

        outputs = attention_model(images)

        preds = torch.argmax(outputs, dim=1)

        all_preds_att.extend(
            preds.cpu().numpy()
        )

        all_labels_att.extend(
            labels.numpy()
        )

# ============================================
# Метрики
# ============================================

attention_accuracy = accuracy_score(
    all_labels_att,
    all_preds_att
)

attention_precision = precision_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

attention_recall = recall_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

attention_f1 = f1_score(
    all_labels_att,
    all_preds_att,
    average="weighted",
    zero_division=0
)

# ============================================
# Сохраняем результаты
# ============================================

attention_results = {
    "Model": "ResNet18 + Self-Attention",
    "Accuracy": attention_accuracy,
    "Precision": attention_precision,
    "Recall": attention_recall,
    "F1-score": attention_f1
}

# ============================================
# Вывод результатов
# ============================================

print("=" * 60)
print("Результаты ResNet18 + Self-Attention")
print("=" * 60)

print(f"Accuracy : {attention_accuracy:.4f}")
print(f"Precision: {attention_precision:.4f}")
print(f"Recall   : {attention_recall:.4f}")
print(f"F1-score : {attention_f1:.4f}")

# ============================================
# Отчет по классам
# ============================================

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels_att,
        all_preds_att,
        zero_division=0
    )
)

# ============================================
# Таблица результатов
# ============================================

attention_results_df = pd.DataFrame(
    [attention_results]
)

print("\nИтоговая таблица:")

display(attention_results_df)

Testing Attention Model: 100%|██████████| 18/18 [00:06<00:00,  2.97it/s]


Результаты ResNet18 + Self-Attention
Accuracy : 0.8986
Precision: 0.9083
Recall   : 0.8986
F1-score : 0.8990

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.92      0.89        13
           1       0.70      0.88      0.78        16
           2       0.82      0.64      0.72        14
           3       1.00      0.92      0.96        12
           4       0.73      0.92      0.81        12
           5       0.75      0.86      0.80        14
           6       0.69      0.90      0.78        10
           7       0.91      1.00      0.95        10
           8       0.76      0.87      0.81        15
           9       0.92      0.86      0.89        14
          10       0.92      0.92      0.92        12
          11       0.94      0.88      0.91        17
          12       1.00      0.75      0.86        12
          13       0.92      1.00      0.96        11
          14       0.92      0.85      0.88        13
 

,Model,Accuracy,Precision,Recall,F1-score
0,ResNet18 + Self-Attention,0.898551,0.908292,0.898551,0.898998
